# Stage 8 (Optional Extension) - Price the customer product

**Input:** only files that were genuinely available on the futures quote date, 29 September 2023 -
the forecast load (`data/forecast_load.csv`, Stage 1), the HPFC (`data/hpfc.csv`, Stage 2), and the
hedge positions/energy for all three strategies (`data/hedge_positions.csv`,
`data/hedge_energy.csv`, `data/selected_products.csv`, Stage 3).

**Deliberately not used:** `data/day_ahead_prices.csv`, `data/imbalance_prices.csv`,
`data/actual_portfolio_load.csv` - all three are *realised 2024* series and were not observable
when the offer would have been made. Using them here would be look-ahead bias.

**Task:** the manager asks what fixed energy price (EUR/MWh) to offer the 2024 customer portfolio.
Develop a transparent pricing approach using only ex-ante information, decide which cost and risk
components belong in the price, state what additional information or assumptions would be needed,
and explain how the recommended hedge (COARSE_CAL) affects the result. Keep the ex-ante price
strictly separate from any conclusion that relies on realised 2024 data.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

DATA = "../data"
DT = 0.25
STRATEGIES = ["UNHEDGED", "COARSE_CAL", "GRANULAR"]
COLORS = {"UNHEDGED": "#8172B3", "COARSE_CAL": "#DD8452", "GRANULAR": "#4C72B0"}
HEDGE_COL = {"UNHEDGED": "hedge_unhedged_mwh", "COARSE_CAL": "hedge_coarse_cal_mwh", "GRANULAR": "hedge_granular_mwh"}

forecast = pd.read_csv(f"{DATA}/forecast_load.csv")
hpfc     = pd.read_csv(f"{DATA}/hpfc.csv")
energy   = pd.read_csv(f"{DATA}/hedge_energy.csv")
positions = pd.read_csv(f"{DATA}/hedge_positions.csv")
selected  = pd.read_csv(f"{DATA}/selected_products.csv")

df = forecast.merge(hpfc, on="timestamp_utc").rename(columns={"total": "L"})
df = df.merge(energy[["timestamp_utc"] + list(HEDGE_COL.values())], on="timestamp_utc")
print("merged grid:", df.shape, "-> forecast load (Stage 1) x HPFC (Stage 2) x hedge energy (Stage 3)")
print("all three inputs were fixed on or before 29 Sep 2023 - nothing here depends on how 2024 turned out")

merged grid: (35136, 11) -> forecast load (Stage 1) x HPFC (Stage 2) x hedge energy (Stage 3)
all three inputs were fixed on or before 29 Sep 2023 - nothing here depends on how 2024 turned out


## 1. What was known on 29 September 2023, and what was not

A transparent price can only use information that genuinely existed at the offer date.

In [2]:
info = pd.DataFrame({
    "Known ex-ante (usable for pricing)": [
        "Customer table, SLP profiles (Stage 1)",
        "Futures snapshot + market_activity (Stage 2)",
        "Historical 2019-2022 price *shape* (Stage 2)",
        "HPFC, hedge optimisation, positions (Stage 2-3)",
    ],
    "Known only ex-post (2024, not usable)": [
        "actual_portfolio_load.csv - realised consumption",
        "day_ahead_prices.csv - realised Day-Ahead prices",
        "imbalance_prices.csv - realised reBAP",
        "All Stage 4-7 results (cost, saving, Dec event)",
    ],
})
info

,Known ex-ante (usable for pricing),"Known only ex-post (2024, not usable)"
0,"Customer table, SLP profiles (Stage 1)",actual_portfolio_load.csv - realised consumption
1,Futures snapshot + market_activity (Stage 2),day_ahead_prices.csv - realised Day-Ahead prices
2,Historical 2019-2022 price *shape* (Stage 2),imbalance_prices.csv - realised reBAP
3,"HPFC, hedge optimisation, positions (Stage 2-3)","All Stage 4-7 results (cost, saving, Dec event)"


## 2. Base procurement cost - fully ex-ante, no assumption needed

The HPFC (Stage 2) already prices every quarter-hour of 2024 using only information available on
29 September 2023: traded Base/Peak quotes and a *historical* price shape. Valuing the forecast
load at the HPFC therefore gives an ex-ante, market-consistent cost estimate that requires no
knowledge of what 2024 actually did:

$$P_{base} = \frac{\sum_t L_t \cdot HPFC_t}{\sum_t L_t}$$

In [3]:
L_total = df["L"].sum()
load_value = (df["L"] * df["hpfc_eur_mwh"]).sum()
P_base = load_value / L_total

print(f"Forecast energy (Stage 1):     {L_total:,.2f} MWh")
print(f"HPFC value of the load:        {load_value:,.2f} EUR")
print(f"Load-weighted average HPFC:    {P_base:,.2f} EUR/MWh   <- this is P_base")

Forecast energy (Stage 1):     43,000.00 MWh
HPFC value of the load:        3,493,705.92 EUR
Load-weighted average HPFC:    81.25 EUR/MWh   <- this is P_base


## 3. Residual shape risk - the size is known ex-ante, its EUR cost is not

The recommended hedge, COARSE_CAL, is value-neutral against the HPFC, but it does not track the
load shape exactly. The size of that mismatch is fully computable now, from Stage 3 outputs alone.

In [4]:
rows = []
for s in STRATEGIES:
    H = df[HEDGE_COL[s]]
    R = df["L"] - H
    rows.append({
        "strategy": s,
        "hedge_notional_MWh": H.sum(),
        "abs_residual_MWh": R.abs().sum(),
        "abs_residual_pct_of_load": 100 * R.abs().sum() / L_total,
        "residual_RMSE_MWh_per_qh": np.sqrt((R ** 2).mean()),
        "residual_HPFC_value_EUR": (R * df["hpfc_eur_mwh"]).sum(),
    })
shape_risk = pd.DataFrame(rows).set_index("strategy").reindex(STRATEGIES)

assert np.allclose(shape_risk.loc[["COARSE_CAL", "GRANULAR"], "residual_HPFC_value_EUR"], 0, atol=1e-3), \
    "value-neutral strategies must price the residual at exactly zero under the HPFC"
print("Check passed: for COARSE_CAL and GRANULAR, the HPFC-value of the residual is 0.00 EUR by")
print("construction (value neutrality) - the shape mismatch is a volume risk, not a value gap.")
shape_risk

Check passed: for COARSE_CAL and GRANULAR, the HPFC-value of the residual is 0.00 EUR by
construction (value neutrality) - the shape mismatch is a volume risk, not a value gap.


,hedge_notional_MWh,abs_residual_MWh,abs_residual_pct_of_load,residual_RMSE_MWh_per_qh,residual_HPFC_value_EUR
strategy,,,,,
UNHEDGED,0.00,"43,000.00",100.00,1.32,"3,493,705.92"
COARSE_CAL,"44,503.82","12,760.13",29.67,0.44,0.00
GRANULAR,"44,079.18","11,985.15",27.87,0.42,0.00


**What this means for pricing.** The residual volume (about 29.67% of load for COARSE_CAL) still
has to be bought or sold on the Day-Ahead market once 2024 arrives, at whatever price then
prevails. That is a real risk the retailer is taking on, and a transparent price should include a
premium for it. But *sizing that premium in EUR/MWh* requires the **absolute historical volatility**
of Day-Ahead prices around the HPFC shape - and `shape_factors.csv` only supplies a *normalised*
shape (annual mean 1), not historical EUR/MWh levels. This is a genuine information gap: without a
historical absolute price series (e.g. 2019-2022 Day-Ahead prices in EUR/MWh, not just their shape),
this premium cannot be derived from the data provided here and would have to come from the retailer's
risk desk.

## 4. Volume / forecast risk - no ex-ante information available at all

Stage 5 quantifies imbalance risk using `imbalance_prices.csv` and `actual_portfolio_load.csv` -
both realised 2024 series. On 29 September 2023 neither existed yet. Nothing in the case data
provided for this exercise gives a historical reBAP series or a historical forecast-error record
that could be used to estimate this premium ex-ante. This is the largest information gap of the
four components: it cannot be sized from the data at all, only assumed from outside experience
(e.g. a retailer's own multi-year settlement history).

## 5. Trading costs and margin

Consistent with the limitation already noted in the main report (Section 4.2): transaction costs,
bid-ask spreads and collateral funding are not modelled anywhere in this case study and would need
to be added from the retailer's own cost accounting. The commercial margin (profit, overheads,
customer credit risk) is a business decision, not something derivable from market data.

## 6. A transparent pricing formula

$$P_{fixed} = P_{base} + \pi_{shape} + \pi_{volume} + \pi_{trading} + \text{margin}$$

Only $P_{base}$ follows from the data. The other four terms are named explicitly rather than
guessed; the numbers below are **illustrative placeholders** to show how the formula combines, not
a claim about what they should be.

In [5]:
P_base_val   = P_base
pi_shape     = 4.00     # ILLUSTRATIVE - needs historical absolute DA volatility (Section 3)
pi_volume    = 3.00     # ILLUSTRATIVE - needs historical reBAP / forecast-error data (Section 4)
pi_trading   = 0.75     # ILLUSTRATIVE - transaction costs / collateral funding (Section 5)
margin       = 4.00     # ILLUSTRATIVE - commercial margin, business decision (Section 5)

price_build = pd.DataFrame({
    "component": ["P_base (data-derived)", "pi_shape (assumed)", "pi_volume (assumed)",
                  "pi_trading (assumed)", "margin (business decision)"],
    "EUR/MWh": [P_base_val, pi_shape, pi_volume, pi_trading, margin],
})
price_build.loc["TOTAL"] = ["P_fixed (illustrative)", price_build["EUR/MWh"].sum()]
price_build

,component,EUR/MWh
0,P_base (data-derived),81.25
1,pi_shape (assumed),4.00
2,pi_volume (assumed),3.00
3,pi_trading (assumed),0.75
4,margin (business decision),4.00
TOTAL,P_fixed (illustrative),93.00


**Read this table as a structure, not a result.** Row 1 (81.25 EUR/MWh) is the only entry this
report can actually defend from the data. The illustrative total of roughly 93 EUR/MWh only shows
how the pieces would stack up if those assumptions were confirmed by the retailer's risk desk.

## 7. How the recommended hedge changes the answer

In [6]:
# Expected procurement cost PER MWh OF LOAD (not per MWh of hedge notional - hedge and load
# volumes differ by exactly the residual, so dividing by hedge notional would mix the Section-3
# shape mismatch back into what should be a pure price comparison). For COARSE_CAL/GRANULAR,
# C_futures = load_value by value neutrality; for UNHEDGED, the HPFC is itself the unbiased
# ex-ante price forecast, so the expected cost of buying unhedged is the same load valuation.
compare = pd.DataFrame({"P_base (EUR/MWh)": {s: P_base_val for s in STRATEGIES}})
compare["abs_residual_pct_of_load"] = shape_risk["abs_residual_pct_of_load"]
compare["monthly_price_std_EUR_per_MWh (Stage 6)"] = [20.15, 2.43, 18.56]
compare

,P_base (EUR/MWh),abs_residual_pct_of_load,monthly_price_std_EUR_per_MWh (Stage 6)
UNHEDGED,81.25,100.00,20.15
COARSE_CAL,81.25,29.67,2.43
GRANULAR,81.25,27.87,18.56


**The expected price is identical across strategies; only the risk around it differs.** This is not
approximate - it follows exactly from the value-neutrality constraint in the Stage 3 hedge
optimisation ($\sum_t H_t \cdot HPFC_t = \sum_t L_t \cdot HPFC_t$): whatever the hedge decision,
the *expected* procurement cost per MWh of load is 81.25 EUR/MWh, because the HPFC is the ex-ante
unbiased price forecast for every quarter-hour, hedged or not. What differs sharply is how much risk
premium the retailer would need to stack on top to confidently stand behind that expected price for
a year: Stage 6 already showed COARSE_CAL's realised monthly-price volatility is about eight times
lower than GRANULAR's and far below UNHEDGED's. A retailer offering COARSE_CAL-backed supply can
therefore justify a *smaller* $\pi_{shape}$ than one backed by GRANULAR or, especially, by no hedge
at all - the hedge choice buys down the size of the risk premium, not the expected cost itself. This
is the same conclusion as the main report's recommendation (Section 3.7), now applied to the
customer price rather than the internal cost comparison.

## 8. Conclusion

An ex-ante-defensible fixed price for the 2024 portfolio has one solid, data-derived anchor -
**81.25 EUR/MWh**, the forecast load valued at the 29 September 2023 HPFC - plus three components
that this case study's data cannot size (a shape-risk premium, a volume/forecast-risk premium, and
trading costs) and one that is a pure business decision (margin). Any number quoted for those four
components in this notebook is illustrative only. What *is* a firm, data-grounded conclusion is
that choosing COARSE_CAL over GRANULAR or UNHEDGED lets the retailer offer the same central price
with a materially smaller risk premium, because it is the strategy with by far the most stable
realised monthly cost (Stage 6). Any comparison of this ex-ante price against what 2024 actually
cost (Stage 4-7) is a backtest of the pricing decision, not part of the pricing decision itself,
and the two must not be conflated.